# Programación funcional · Iteradores, generadores y decoradores

Protocolo iterador, generadores, evaluación perezosa, `itertools` y decoradores.

## 1. Protocolo iterador

Crear una clase `CuentaAtras` que implemente `__iter__` y `__next__` para recorrer los números desde `inicio` hasta 0.

In [1]:
class CuentaAtras:
    def __init__(self, inicio):
        self.actual = inicio

    def __iter__(self):
        return self

    def __next__(self):
        if self.actual < 0:
            raise StopIteration
        valor = self.actual
        self.actual -= 1
        return valor

print(list(CuentaAtras(5)))

it = iter(CuentaAtras(2))
print(next(it), next(it), next(it), next(it, "fin"))
assert list(CuentaAtras(3)) == [3, 2, 1, 0]

[5, 4, 3, 2, 1, 0]
2 1 0 fin


## 2. Generador de Fibonacci

Escribir un generador infinito con `yield` que produzca la sucesión de Fibonacci y obtener los 10 primeros términos con `itertools.islice`.

In [2]:
from itertools import islice

def fibonacci():
    a, b = 0, 1
    while True:
        yield a
        a, b = b, a + b

gen = fibonacci()
print(next(gen), next(gen), next(gen))
print(list(islice(fibonacci(), 10)))
assert list(islice(fibonacci(), 10)) == [0, 1, 1, 2, 3, 5, 8, 13, 21, 34]

0 1 1
[0, 1, 1, 2, 3, 5, 8, 13, 21, 34]


## 3. Iterable reutilizable

Crear una clase `RangoDecimal(inicio, fin, paso)` cuyo `__iter__` sea un generador, de modo que pueda recorrerse varias veces. Comparar con `CuentaAtras`, que se agota tras el primer recorrido.

In [3]:
class RangoDecimal:
    def __init__(self, inicio, fin, paso):
        self.inicio, self.fin, self.paso = inicio, fin, paso

    def __iter__(self):
        n = 0
        # multiplicar evita acumular error de redondeo
        while (valor := round(self.inicio + n * self.paso, 10)) < self.fin:
            yield valor
            n += 1

rango = RangoDecimal(0, 1, 0.1)
print(list(rango))
print(sum(rango))

cuenta = CuentaAtras(3)
print(list(cuenta), list(cuenta))
assert list(rango) == list(rango)

[0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
4.5
[3, 2, 1, 0] []


## 4. `yield from`

Escribir un generador `aplana` que recorra una lista con listas y tuplas anidadas a cualquier profundidad y produzca sus elementos uno a uno, delegando con `yield from`.

In [4]:
def aplana(elementos):
    for elemento in elementos:
        if isinstance(elemento, (list, tuple)):
            yield from aplana(elemento)
        else:
            yield elemento

anidada = [1, [2, 3, [4, (5, 6)]], [], 7, [[[8]]]]
print(list(aplana(anidada)))
assert list(aplana(anidada)) == [1, 2, 3, 4, 5, 6, 7, 8]

[1, 2, 3, 4, 5, 6, 7, 8]


## 5. Expresiones generadoras y evaluación perezosa

Comparar la memoria que ocupa una lista por comprensión con la de una expresión generadora equivalente. Comprobar que un generador solo calcula los valores cuando se le piden y que se agota tras recorrerlo.

In [5]:
import sys

cuadrados_lista = [n * n for n in range(1_000_000)]
cuadrados_gen = (n * n for n in range(1_000_000))
print(f"lista:     {sys.getsizeof(cuadrados_lista):>9} bytes")
print(f"generador: {sys.getsizeof(cuadrados_gen):>9} bytes")

print(sum(cuadrados_gen))
print(sum(cuadrados_gen))  # ya agotado

lista:       8448728 bytes
generador:       200 bytes
333332833333500000
0


In [6]:
def numeros():
    for n in range(1, 6):
        print(f"  genero {n}")
        yield n

pares = (n for n in numeros() if n % 2 == 0)
print("todavía no se ha generado nada")
print("primer par:", next(pares))

primo = next(n for n in range(10_000, 20_000) if all(n % d for d in range(2, int(n ** 0.5) + 1)))
print("primer primo desde 10000:", primo)
assert primo == 10007

todavía no se ha generado nada
  genero 1
  genero 2
primer par: 2
primer primo desde 10000: 10007


## 6. Tubería de generadores

Crear un fichero de log y procesarlo con una cadena de generadores: leer líneas, convertirlas en diccionarios y quedarse con las de nivel `ERROR`, sin cargar el fichero entero en memoria.

In [7]:
from pathlib import Path

Path("servidor.log").write_text(
    "2025-03-01 08:00:01 INFO arranque del servidor\n"
    "2025-03-01 08:05:12 ERROR conexión rechazada desde 10.0.0.7\n"
    "2025-03-01 08:05:13 WARNING reintento de conexión\n"
    "2025-03-01 09:17:40 ERROR disco al 95 %\n"
    "2025-03-01 10:00:00 INFO copia de seguridad completada\n",
    encoding="utf-8",
)

def lee_lineas(ruta):
    with open(ruta, encoding="utf-8") as f:
        for linea in f:
            yield linea.rstrip("\n")

def parsea(lineas):
    for linea in lineas:
        fecha, hora, nivel, mensaje = linea.split(" ", 3)
        yield {"hora": hora, "nivel": nivel, "mensaje": mensaje}

def filtra(registros, nivel):
    return (r for r in registros if r["nivel"] == nivel)

errores = list(filtra(parsea(lee_lineas("servidor.log")), "ERROR"))
for r in errores:
    print(r["hora"], r["mensaje"])
assert len(errores) == 2

08:05:12 conexión rechazada desde 10.0.0.7
09:17:40 disco al 95 %


## 7. `itertools`

Resolver con `itertools`: unir listas (`chain`), saldo acumulado (`accumulate`), agrupar palabras por inicial (`groupby`), parejas de un torneo (`combinations`), variantes de un producto (`product`), diferencias entre días consecutivos (`pairwise`), lotes (`batched`) y turnos rotatorios (`cycle`).

In [8]:
from itertools import accumulate, batched, chain, combinations, cycle, groupby, islice, pairwise, product

print(list(chain([1, 2], (3, 4), range(5, 7))))

movimientos = [100, -30, -20, 50, -80]
print(list(accumulate(movimientos)))
assert list(accumulate(movimientos))[-1] == 20

palabras = sorted(["gofio", "guagua", "papas", "mojo", "millo", "pella"])
print({inicial: list(grupo) for inicial, grupo in groupby(palabras, key=lambda p: p[0])})

equipos = ["Tenerife", "Las Palmas", "Lanzarote", "La Palma"]
partidos = list(combinations(equipos, 2))
print(len(partidos), partidos[:3])
assert len(partidos) == 6

[1, 2, 3, 4, 5, 6]
[100, 70, 50, 100, 20]
{'g': ['gofio', 'guagua'], 'm': ['millo', 'mojo'], 'p': ['papas', 'pella']}
6 [('Tenerife', 'Las Palmas'), ('Tenerife', 'Lanzarote'), ('Tenerife', 'La Palma')]


In [9]:
variantes = [f"{talla}-{color}" for talla, color in product("SML", ["rojo", "azul"])]
print(variantes)

temperaturas = [21, 23, 22, 25, 24]
print([b - a for a, b in pairwise(temperaturas)])

print(list(batched(range(1, 11), 3)))

turnos = cycle(["Ana", "Luis", "Marta"])
print([f"día {d}: {persona}" for d, persona in zip(range(1, 6), turnos)])
assert list(islice(cycle("AB"), 5)) == list("ABABA")

['S-rojo', 'S-azul', 'M-rojo', 'M-azul', 'L-rojo', 'L-azul']
[2, -1, 3, -1]
[(1, 2, 3), (4, 5, 6), (7, 8, 9), (10,)]
['día 1: Ana', 'día 2: Luis', 'día 3: Marta', 'día 4: Ana', 'día 5: Luis']


## 8. Decorador simple

Escribir un decorador `cuenta_llamadas` que guarde en un atributo de la función cuántas veces se ha llamado.

In [10]:
def cuenta_llamadas(funcion):
    def envoltura(*args, **kwargs):
        envoltura.llamadas += 1
        return funcion(*args, **kwargs)
    envoltura.llamadas = 0
    return envoltura

@cuenta_llamadas
def saluda(nombre):
    return f"hola {nombre}"

print(saluda("Ana"), saluda("Luis"), saluda("Marta"))
print("llamadas:", saluda.llamadas)
assert saluda.llamadas == 3

hola Ana hola Luis hola Marta
llamadas: 3


## 9. `functools.wraps`

Escribir un decorador `registra` que muestre los argumentos y el resultado de cada llamada. Comprobar que sin `functools.wraps` se pierden el nombre y la documentación de la función original.

In [11]:
from functools import wraps

def registra_sin_wraps(funcion):
    def envoltura(*args, **kwargs):
        return funcion(*args, **kwargs)
    return envoltura

def registra(funcion):
    @wraps(funcion)
    def envoltura(*args, **kwargs):
        resultado = funcion(*args, **kwargs)
        print(f"{funcion.__name__}{args} → {resultado}")
        return resultado
    return envoltura

@registra_sin_wraps
def area(base, altura):
    """Área de un triángulo."""
    return base * altura / 2

@registra
def perimetro(a, b, c):
    """Perímetro de un triángulo."""
    return a + b + c

print(area.__name__, area.__doc__)
print(perimetro.__name__, perimetro.__doc__)
perimetro(3, 4, 5)
assert perimetro.__name__ == "perimetro"

envoltura None
perimetro Perímetro de un triángulo.
perimetro(3, 4, 5) → 12


## 10. Decoradores con argumentos

Escribir un decorador `repite(veces)` que ejecute la función varias veces y devuelva la lista de resultados, y otro `solo_tipos(*tipos)` que lance `TypeError` si algún argumento posicional no es de los tipos indicados.

In [12]:
import random
from functools import wraps

def repite(veces):
    def decorador(funcion):
        @wraps(funcion)
        def envoltura(*args, **kwargs):
            return [funcion(*args, **kwargs) for _ in range(veces)]
        return envoltura
    return decorador

def solo_tipos(*tipos):
    def decorador(funcion):
        @wraps(funcion)
        def envoltura(*args):
            for arg in args:
                if not isinstance(arg, tipos):
                    raise TypeError(f"{funcion.__name__}: {arg!r} no es de tipo {tipos}")
            return funcion(*args)
        return envoltura
    return decorador

random.seed(1)

@repite(5)
def tira_dado():
    return random.randint(1, 6)

@solo_tipos(int, float)
def suma(*numeros):
    return sum(numeros)

print(tira_dado())
print(suma(1, 2.5, 3))
try:
    suma(1, "2")
except TypeError as e:
    print("TypeError:", e)
assert len(tira_dado()) == 5

[2, 5, 1, 3, 1]
6.5
TypeError: suma: '2' no es de tipo (<class 'int'>, <class 'float'>)


## 11. Memoización y temporizador

Escribir un decorador `memoriza` que guarde en un diccionario los resultados ya calculados y otro `temporiza` que muestre cuánto tarda una función. Compararlos con Fibonacci recursivo.

In [13]:
import time
from functools import wraps

def temporiza(funcion):
    @wraps(funcion)
    def envoltura(*args, **kwargs):
        inicio = time.perf_counter()
        resultado = funcion(*args, **kwargs)
        print(f"{funcion.__name__}: {time.perf_counter() - inicio:.4f} s")
        return resultado
    return envoltura

def memoriza(funcion):
    cache = {}

    @wraps(funcion)
    def envoltura(*args):
        if args not in cache:
            cache[args] = funcion(*args)
        return cache[args]

    envoltura.cache = cache
    return envoltura

def fib(n):
    return n if n < 2 else fib(n - 1) + fib(n - 2)

@memoriza
def fib_memo(n):
    return n if n < 2 else fib_memo(n - 1) + fib_memo(n - 2)

@temporiza
def fibonacci_lento(n):
    return fib(n)

@temporiza
def fibonacci_memorizado(n):
    return fib_memo(n)

print(fibonacci_lento(27), fibonacci_memorizado(27))
print("valores en caché:", len(fib_memo.cache))
assert fib_memo(80) == 23416728348467685

fibonacci_lento: 0.0192 s
fibonacci_memorizado: 0.0000 s
196418 196418
valores en caché: 28


## 12. `functools.lru_cache`

Calcular de forma recursiva cuántos caminos hay desde la esquina superior izquierda hasta la inferior derecha de una cuadrícula moviéndose solo a la derecha o hacia abajo, usando `functools.lru_cache`. Mostrar las estadísticas de la caché.

In [14]:
from functools import lru_cache
from math import comb

@lru_cache(maxsize=None)
def caminos(filas, columnas):
    if filas == 1 or columnas == 1:
        return 1
    return caminos(filas - 1, columnas) + caminos(filas, columnas - 1)

print(caminos(3, 3), caminos(18, 18))
print(caminos.cache_info())
assert caminos(18, 18) == comb(34, 17)

caminos.cache_clear()
print(caminos.cache_info())

6 2333606220
CacheInfo(hits=257, misses=323, maxsize=None, currsize=323)
CacheInfo(hits=0, misses=0, maxsize=None, currsize=0)
